<a href="https://colab.research.google.com/github/Nebius-Academy/LLM-Engineering-Essentials/blob/main/topic2/2.2_llm_workflows.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Открыть в Colab"/></a>

# LLM Engineering Essentials от Nebius Academy
Курс на github: [ссылка](https://github.com/Nebius-Academy/LLM-Engineering-Essentials/tree/main)
Автор: Алексей Умнов
Ссылки:
- [LinkedIn](www.linkedin.com/in/alex-umnov)
- Профиль в Discord: *alexumnov*, лучше всего отметить #nebius-academy.
Курс сейчас находится в разработке, скоро появятся дополнительные материалы. [Подпишитесь, чтобы быть в курсе](https://academy.nebius.com/llm-engineering-essentials/update/)
# 2.2 Рабочие процессы LLM
В теме 1 мы в основном изучали, как получить выгоду от одного LLM, работающего в режиме одного звонка или чата. Но это только начало! Гораздо большего можно достичь, организовав сложный рабочий процесс, объединяющий несколько вызовов LLM, инструментов и т. д.
Оркестрация будет основной идеей Темы 2. Мы проведем вас через:
* **Рабочие процессы LLM** – **руководство несколькими вызовами LLM внутри одной системы** — в этом блокноте
* Организованные и встроенные процессы рассуждения LLM в блокнотах **2.3–5**
* Использование встроенных инструментов и основы агента LLM в **2.6**.
* Системы планирования и агентирования на основе LLM в **2.7**
Итак, начнем это увлекательное путешествие!
В этих блокнотах мы обсудим, как осмысленно и гибко объединять звонки LLM. В основном мы будем следовать статье [Создание эффективных агентов](https://www.anthropic.com/engineering/building-effective-agents) статьи Anthropic в ее классификации рабочих процессов — и мы настоятельно рекомендуем вам просмотреть ее.

## Готовим вещи

In [ ]:
!pip install openai -qU

In [ ]:
from google.colab import userdata
from openai import OpenAI
import os

os.environ['NEBIUS_API_KEY'] = userdata.get("nebius_api_key")

nebius_client = OpenAI(
    base_url="https://api.studio.nebius.ai/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

llama_model = "meta-llama/Llama-3.3-70B-Instruct"

def prettify_string(text, max_line_length=80):
    """Prints a string with line breaks at spaces to prevent horizontal scrolling.

    Args:
        text: The string to print.
        max_line_length: The maximum length of each line.
    """

    output_lines = []
    lines = text.split("\n")
    for line in lines:
        current_line = ""
        words = line.split()
        for word in words:
            if len(current_line) + len(word) + 1 <= max_line_length:
                current_line += word + " "
            else:
                output_lines.append(current_line.strip())
                current_line = word + " "
        output_lines.append(current_line.strip())  # Append the last line
    return "\n".join(output_lines)

def answer_with_llm(prompt: str,
                    system_prompt="You are a helpful assistant",
                    max_tokens=512,
                    client=nebius_client,
                    model=llama_model,
                    prettify=True,
                    temperature=None) -> str:

    messages = []

    if system_prompt:
        messages.append(
            {
                "role": "system",
                "content": system_prompt
            }
        )

    messages.append(
        {
            "role": "user",
            "content": prompt
        }
    )

    completion = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature
    )

    if prettify:
        return prettify_string(completion.choices[0].message.content)
    else:
        return completion.choices[0].message.content


# Понимание рабочих процессов LLM

## Цепочка
Самый простой тип рабочего процесса LLM — это **цепочка**: использование нескольких вызовов LLM в последовательности, при этом следующий из них изменяет или уточняет предыдущие.
<center>
<img исходник="
https://www.anthropic.com/_next/image?url=https%3A%2F%2Fwww-cdn.anthropic.com%2Fimages%2F4zrzovbb%2Fwebsite%2F7418719e3dab222dccb379b8879e1dc08ad34c78-2401x1000.png&w=3840&q=75" width=1000 />
[Источник](https://www.anthropic.com/engineering/building-effective-agents)
</center>
Примеры вариантов использования могут включать в себя:
* **Локализация**. Хотя LLM развиваются в сторону многоязычия, им по-прежнему легче отвечать на сложные инструкции на английском языке (поскольку большая часть обучающих данных, а также большая часть Интернета и книг находится на английском языке). Естественный способ справиться с этим — создать **цепочку**:
  * Первый вызов LLM переводит запрос с исходного языка на английский.
  * Второй обрабатывает запрос на английском языке
  * Третий переводит ответ обратно на исходный язык.
До того, как LLM стали хорошо справляться со структурированными выводами, другим популярным вариантом использования цепочки был (Ответ на вопрос) -> (Извлечение ответа).
Однако во многих случаях рабочие процессы не являются последовательными, поэтому давайте обсудим несколько более сложных типов.

## Распараллеливание
**Распараллеливание** – это тип рабочего процесса, при котором несколько исполнителей обрабатывают запрос, а их результаты объединяются агрегатором для получения окончательного ответа.
<center>
<img исходник="
https://www.anthropic.com/_next/image?url=https%3A%2F%2Fwww-cdn.anthropic.com%2Fimages%2F4zrzovbb%2Fwebsite%2F406bb032ca007fd1624f261af717d70e6ca86286-2401x1000.png&w=3840&q=75" width=1000 />
[Источник](https://www.anthropic.com/engineering/building-effective-agents)
</center>
Особым случаем распараллеливания является то, что можно назвать **LLM MapReduce**. В качестве примера рассмотрим длинное обобщение документов. если входные данные слишком велики, чтобы их можно было эффективно обработать за один вызов, мы можем
* распределять входные фрагменты между идентичными исполнителями LLM (фаза **карта**)
* затем попросите другого LLM собрать воедино краткое изложение отдельных фрагментов (этап **перезаписи**)
Другим примером может быть **оценка разговоров с чат-ботами**. Обычно вы хотите оценить мастерство вашего чат-бота по нескольким критериям: готовность помочь, тон голоса и т. д. – и все это можно оценить с помощью **LLM-as-a-Judge**. И вообще, было бы неплохо оценивать разные параметры в разных и параллельных вызовах LLM - таким образом промпты будут проще, а результаты судей - более надежными. В такой системе агрегатор не является обязательным; вы можете просто сложить все баллы вместе без каких-либо дополнительных LLM.

## Маршрутизация
Чат-бот службы поддержки клиентов может иметь сложную древовидную логику, переключающую пользователя между несколькими ветвями разговора. Вы можете просто тщательно подсказать один LLM и позволить ему исключить это в режиме чата, но если вы можете описать все сценарии, почему бы не сделать ситуацию более надежной, создав рабочий процесс **маршрутизации**?
<center>
<img исходник="
https://www.anthropic.com/_next/image?url=https%3A%2F%2Fwww-cdn.anthropic.com%2Fimages%2F4zrzovbb%2Fwebsite%2F5c0c0e9fe4def0b584c04d37849941da55e5e71c-2401x1000.png&w=3840&q=75" width=1000 />
[Источник](https://www.anthropic.com/engineering/building-effective-agents)
</center>
В самой базовой реализации LLM точки входа выбирает один из нескольких предписанных сценариев на основе запроса пользователя. Но рабочий процесс может быть и более сложным:
<center>
<img src="https://drive.google.com/uc?export=view&id=1QoIL5j2C6U5u2gE_jGcF-w3e1qlgWL7d" width=600 />
</center>
Не все ссылки должны быть другими LLM. Некоторые из них могут быть обработчиками на основе правил или даже привлекать специалистов по поддержке, которые готовы помочь клиенту.
Другое возможное применение маршрутизации — выбор между несколькими LLM различной мощности. Например, вы можете использовать модель 8B для простых вопросов или модель 70B для более сложных. Классификация сложности вопроса может оказаться задачей для магистра права.

## Петли обратной связи
В сложных задачах мы могли бы ожидать, что рабочий процесс LLM не сразу приведет к окончательному решению. Хорошим примером является программирование: первое решение может быть ошибочным, и может помочь один или два раунда самоанализа и самоисправления.
Если вы можете описать критерии оценки, вы можете построить **цикл обратной связи**, который будет работать до тех пор, пока оценщик не выполнит решение или пока система не достигнет максимального количества итераций.
<center>
<img исходник="
https://www.anthropic.com/_next/image?url=https%3A%2F%2Fwww-cdn.anthropic.com%2Fimages%2F4zrzovbb%2Fwebsite%2F14f51e6406ccb29e695da48b17017e899a6119c7-2401x1000.png&w=3840&q=75" width=1000 />
[Источник](https://www.anthropic.com/engineering/building-effective-agents)
</center>

## Создание более сложных рабочих процессов. Рабочие процессы против агентов
Из этих четырех примитивов можно собрать рабочие процессы произвольной сложности. Например, вот потенциальный рабочий процесс создания рекламы:
<center>
<img src="https://drive.google.com/uc?export=view&id=14DDGHYOojUClbcI59WRn5lZ_4rAWHpoW" width=600 />
</center>
Все рабочие процессы LLM разработаны человеком. Для его создания необходимо придумать узлы процесса и связи между ними. Иногда вам хочется чего-нибудь – получить степень магистра права! - организовать все для вас.
Вот так:
<center>
<img src="https://drive.google.com/uc?export=view&id=102GUavLMfYjqR0SftsQa5VItH2TEA0JS" width=400 />
</center>
Оркестровка на базе LLM превращает систему в **Агента LLM**. Это крутая и мощная вещь, и мы обсудим ее более подробно в блокнотах [A.1](https://colab.research.google.com/github/Nebius-Academy/LLM-Engineering-Essentials/blob/main/topic2/a.1_llm_tools_and_agents.ipynb) и [A.2](https://colab.research.google.com/github/Nebius-Academy/LLM-Engineering-Essentials/blob/main/topic2/a.1_llm_tools_and_agents.ipynb). Однако это делает вещи менее прозрачными и менее надежными по сравнению с конвейерами, оркестрируемыми вручную.
В оставшейся части этой записной книжки мы рассмотрим несколько конкретных примеров оркестровки LLM: обобщение и локализацию.

# Примеры рабочих процессов LLM

## Подведение итогов

Давайте попробуем написать простой сценарий обобщения LLM. В качестве примера текста возьмем статью из Википедии о лапах.
Примечание. Здесь мы возьмем другую модель, а именно **deepseek-ai/DeepSeek-R1**, поскольку ее токенизатор не требует дополнительных лицензионных требований.

In [ ]:
import requests, bs4

content = requests.get("https://en.wikipedia.org/wiki/Paw").content
parsed = bs4.BeautifulSoup(content)
content_div = parsed.find("div", "mw-content-container")
full_text = content_div.get_text()

Мы используем **BeautifulSoup** для анализа содержимого страницы, опуская все, кроме основного текста.
Теперь давайте напишем простой код суммирования llm:

In [ ]:
model = "deepseek-ai/DeepSeek-R1"

def summarize_with_llm(text):
    chat_completion = nebius_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": f"Summarize the most important aspects of the following text: {text}. Try to be short."}]
    )
    return chat_completion.choices[0].message.content

In [ ]:
summarize_with_llm(full_text)

Это выглядит как хорошее резюме статьи. Однако это самый простой пример. Попробуем сделать что-нибудь посложнее.

## Карта сокращает суммирование

In [ ]:
def get_wiki_text(url):
    content = requests.get(url).content
    parsed = bs4.BeautifulSoup(content)
    content_div = parsed.find("div", "mw-content-container")
    full_text = content_div.get_text()
    return full_text

Страница *2023 года на американском телевидении* считается одной из самых длинных страниц в Википедии. В основном он длинный, потому что это список всех шоу, выпущенных в этом году, с описаниями. Тем не менее, нам будет полезно проверить свои навыки резюмирования длинных текстов.

In [ ]:
full_text = get_wiki_text("https://en.wikipedia.org/wiki/2023_in_American_television")

Длина текста нам не так интересна, как количество токенов. Давайте попробуем рассмотреть оба.
Мы воспользуемся **huggingface**, чтобы получить токенизатор модели.

In [ ]:
from transformers import AutoTokenizer


def get_token_count(text):
    encoding = AutoTokenizer.from_pretrained(model)

    encoded = encoding.encode(text)
    return len(encoded)

In [ ]:
print(f"Number of tokens: {get_token_count(full_text)}")
print(f"Number of characters: {len(full_text)}")

Несмотря на то, что технически DeepSeek R1 может принять весь этот текст сразу (он имеет контекстное окно длиной 164 000 токенов), обычно это не идеально. Тем более, что для моделей вычисления растут более чем линейно пропорционально длине. Поэтому лучше суммировать в стиле **MapReduce**, когда мы суммируем небольшие части, а затем генерируем сводку для всего текста. Это также может помочь нам суммировать тексты, размер которых превышает наше контекстное окно.
Давайте поэкспериментируем с обоими:

In [ ]:
naive_summary = summarize_with_llm(full_text)
naive_summary

Чтобы организовать конвейер MapReduce, нам нужно разбить текст на куски, и мы не хотим, чтобы он был разрезан в середине предложения. Итак, мы будем использовать для этого специальные инструменты.
В Langchain есть удобный инструмент под названием `TextSplitter`, который позволяет разбивать текст по определенным правилам.
Например, `RecursiveCharacterTextSplitter` может рекурсивно разбивать текст на основе списка символов, пока он не достигнет желаемой длины. Это также позволяет вам настроить перекрытие, чтобы фрагменты имели некоторые связи между собой. Это полезная вещь, но мы не будем здесь ее использовать.
Список разделителей по умолчанию: `["\n\n", "\n", " ", ""]`. Что теоретически дает нам разделение по абзацам, подпунктам, словам, а затем по символам.
Langchain также позволяет вам определять функции длины для разделителей, которые будут использоваться для определения того, имеет ли фрагмент подходящую длину. Мы даже можем напрямую создать экземпляр функции длины из кодера `tiktoken` так, чтобы длина нашего фрагмента была привязана к количеству токенов.

In [ ]:
!pip install -qU langchain langchain-openai

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=AutoTokenizer.from_pretrained(model),
    chunk_size=10000,
    chunk_overlap=0
)

In [ ]:
splitted_text = splitter.split_text(full_text)
len(splitted_text)

Давайте создадим операции Map и уменьшить.
Обратите внимание, что langchain использует немного другое обозначение для своих цепочек.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model=llama_model,
    base_url="https://api.studio.nebius.ai/v1/",
    api_key=os.environ['NEBIUS_API_KEY']
)

map_prompt = ChatPromptTemplate.from_messages(
    [("human", "Write a concise summary of the following:\\n\\n{context}")]
)

map_chain = map_prompt | llm | StrOutputParser()


reduce_template = """
The following is a set of summaries:
{docs}
Take these and distill it into a final, consolidated summary
of the main themes.
"""

reduce_prompt = ChatPromptTemplate([("human", reduce_template)])

reduce_chain = reduce_prompt | llm | StrOutputParser()

Теперь в простейшей форме наше обобщение MapReduce будет выглядеть так:

In [ ]:
from tqdm.auto import tqdm

In [ ]:
def map_reduce_summarization(docs):
    summaries = map(
        lambda doc: map_chain.invoke({"context": doc}),
        tqdm(docs)
    )

    final_summary = reduce_chain.invoke({"docs": "\n\n".join(summaries)})
    return final_summary

In [ ]:
map_reduce_summarization(splitted_text)

## Уменьшение карты с помощью оркестрации LLM
В некоторых случаях нам может потребоваться более сложная оркестровка вызовов MapReduce. Давайте представим себе примерный сценарий.
Мы хотим создать партию искателей приключений. Мы можем разбить эту задачу на несколько шагов. А именно:
1. Составьте приблизительный план партии, количество членов, приблизительные описания.
2. Для каждого участника создайте полную историю и список навыков.
3. Соберите все описания и составьте небольшой рассказ о том, как они оказались вместе.
<center>
<img src="https://drive.google.com/uc?export=view&id=1ScixZwB5AK9TQWrpIg5nTErT_M0Y-S9C" width=600 />
</center>
Для части 2 мы можем повторно использовать наш предыдущий пример из блокнота структурированной генерации.

In [ ]:
from typing import List
from pydantic import BaseModel

class CharacterProfile(BaseModel):
    name: str
    age: int
    special_skills: List[str]
    traits: List[str]
    character_class: str
    origin: str

def generate_character(description: str):
    completion = nebius_client.beta.chat.completions.parse(
        model=llama_model,
        messages=[
            {
                "role": "user",
                "content": "Design a role play character based on the following"\
                          f"short description {description}"}
        ],
        extra_body={
            "guided_json": CharacterProfile.model_json_schema()
        }
    )

    return CharacterProfile.model_validate_json(completion.choices[0].message.content)

In [ ]:
generate_character("Dwarf druid")

Теперь о шагах 1 и 3:

In [ ]:
import json

def pregenerate_party():
    json_output = nebius_client.chat.completions.create(
        messages=[{
            'role': 'user', \
            'content': 'Generate a short description for a party of adventurers.\n'\
            'A party should have 3-5 adventures and a balanced classes set, i.e. '\
            'have at least a melee tank, a support and a damage dealer. \n'\
            'Output those short descriptions in a json format as a list with the key "party".\n'\
            'Each description should be a string with only a couple of details'

        }],
        model=llama_model,
        response_format={"type": "json_object"}
    ).choices[0].message.content
    return json.loads(json_output)['party']

In [ ]:
pregenerate_party()

In [ ]:
def generate_back_story(party_details: str):
    return nebius_client.chat.completions.create(
        messages=[
            {
                'role': 'user', \
                'content': 'Based on the following party details generate '\
                'a short story of how this party came to be together.\n'
                f'{str(party_details)}'
            }
        ],
        model=llama_model,
    ).choices[0].message.content

Теперь объединим все это в рабочий процесс:

In [ ]:
def generate_party():
    party = pregenerate_party()

    character_sheets = [
        generate_character(character)
        for character in party
    ]

    backstory = generate_back_story(character_sheets)

    character_sheets_str = "\n".join([
        str(character) for character in character_sheets
    ])

    return f"""
Party description:
{json.dumps(party, indent=4)}

Party backstory:
{backstory}

Party members:
{character_sheets_str}
"""

In [ ]:
print(generate_party())

## LLM-локализация

Даже если ваш базовый LLM лучше владеет английским, чем целевым языком, вы можете легко перевести свои результаты с помощью другого звонка LLM.

In [ ]:
def translate_to_language(input: str, target_language: str):
    return nebius_client.chat.completions.create(
        messages=[
            {
                'role': 'user', \
                'content': f'Translate the following text into {target_language}:\n{input}'
            }
        ],
        model=llama_model,
    ).choices[0].message.content

In [ ]:
party = generate_party()
print(party)
translated_party = translate_to_language(party, "Spanish")
print(translated_party)

Давайте также попробуем попросить LLM сгенерировать партию на испанском языке, минуя этап перевода:

In [ ]:
def pregenerate_party_in_language(target_language: str):
    json_output = nebius_client.chat.completions.create(
        messages=[{
            'role': 'user', \
            'content': 'Generate a short description for a party of adventurers.\n'\
            'A party should have 3-5 adventures and a balanced classes set, i.e. '\
            'have at least a melee tank, a support and a damage dealer. \n'\
            'Output those short descriptions in a json format as a list with the key "party".\n'\
            'Each description should be a string with only a couple of details.\n'\
            f'Generate in {target_language}'
        }],
        model=llama_model,
        response_format={"type": "json_object"}
    ).choices[0].message.content
    return json.loads(json_output)['party']

In [ ]:
pregenerate_party_in_language("Dutch")

# Практические задания

## Задача 1. Локализация символов
Давайте добавим локализацию в наш простой класс NPC-чата из темы 1.
Ваша задача будет заключаться в реализации следующего конвейера локализованного чата:
- Ввод пользователя переводится на английский язык,
- NPC отвечает на английский запрос на английском языке (уже реализовано)
- Ответ NPC переводится на целевой язык, и перевод возвращается пользователю.
Вот весь код для создания персонажа, который мы использовали ранее. Добавьте новый код в нужные места

In [ ]:
from collections import defaultdict, deque
from openai import OpenAI
from typing import Dict, Any, Optional
import datetime
import string
import random
from dataclasses import dataclass

@dataclass
class NPCConfig:
    world_description: str
    character_description: str
    history_size: int = 10
    has_scratchpad: bool = False

class NPCFactoryError(Exception):
    """Base exception class for NPC Factory errors."""
    pass

class NPCNotFoundError(NPCFactoryError):
    """Raised when trying to interact with a non-existent NPC."""
    def __init__(self, npc_id: str):
        self.npc_id = npc_id
        super().__init__(f"NPC with ID '{npc_id}' not found")

class SimpleChatNPC:
    def __init__(self, client: OpenAI, model: str, config: NPCConfig):
        self.client = client
        self.model = model
        self.config = config
        self.chat_histories = defaultdict(lambda: deque(maxlen=config.history_size))

    def get_system_message(self) -> Dict[str, str]:
        """Returns the system message that defines the NPC's behavior."""
        character_description = self.config.character_description

        if self.config.has_scratchpad:
            character_description += """
You can use scratchpad for thinking before you answer: whatever you output between #SCRATCHPAD and #ANSWER won't be shown to anyone.
You start your output with #SCRATCHPAD and after you've done thinking, you #ANSWER"""

        return {
            "role": "system",
            "content": f"""WORLD SETTING: {self.config.world_description}
###
{character_description}"""
        }

    def chat(self, user_message: str, user_id: str) -> str:
        """Process a user message and return the NPC's response."""
        messages = [self.get_system_message()]

        # Add conversation history
        history = list(self.chat_histories[user_id])
        if history:
            messages.extend(history)

        # Add new user message
        user_message_dict = {
            "role": "user",
            "content": user_message
        }
        self.chat_histories[user_id].append(user_message_dict)
        messages.append(user_message_dict)

        try:
            completion = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=0.6
            )

            response = completion.choices[0].message.content

            # Handle scratchpad if enabled
            response_clean = response
            if self.config.has_scratchpad:
                import re
                scratchpad_match = re.search(r"#SCRATCHPAD(:?)(.*?)#ANSWER(:?)", response, re.DOTALL)
                if scratchpad_match:
                    response_clean = response[scratchpad_match.end():].strip()


            # Store response in history, including the scratchpad
            self.chat_histories[user_id].append({
                "role": "assistant",
                "content": response
            })

            # Return the message to the user without a scratchpad
            return response_clean

        except Exception as e:
            return f"Error: {str(e)}"

class NPCFactory:
    def __init__(self, client: OpenAI, model: str):
        self.client = client
        self.model = model
        self.npcs: Dict[str, SimpleChatNPC] = {}
        self.user_ids: Dict[str, str] = {}  # username -> user_id mapping

    def generate_id(self) -> str:
        """Generate a random unique identifier."""
        return ''.join(random.choice(string.ascii_letters) for _ in range(8))

    def register_user(self, username: str) -> str:
        """Register a new user and return their unique ID.
        If username already exists, appends a numerical suffix."""
        base_username = username
        suffix = 1

        # Keep trying with incremented suffixes until we find an unused name
        while username in self.user_ids:
            username = f"{base_username}_{suffix}"
            suffix += 1

        user_id = self.generate_id()
        self.user_ids[username] = user_id
        return user_id

    def set_user_language(self, user_id: str, language: str):
        """Set the preferred language for a user."""
        ### YOUR CODE HERE

    def register_npc(self, world_description: str, character_description: str,
                     history_size: int = 10, has_scratchpad: bool = False) -> str:
        """Create and register a new NPC, returning its unique ID."""
        npc_id = self.generate_id()

        config = NPCConfig(
            world_description=world_description,
            character_description=character_description,
            history_size=history_size,
            has_scratchpad=has_scratchpad
        )

        self.npcs[npc_id] = SimpleChatNPC(self.client, self.model, config)
        return npc_id

    def chat_with_npc(self, npc_id: str, user_id: str, message: str) -> str:
        """Send a message to a specific NPC from a specific user.

        Args:
            npc_id: The unique identifier of the NPC
            user_id: The unique identifier of the user
            message: The message to send

        Returns:
            The NPC's response

        Raises:
            NPCNotFoundError: If the specified NPC doesn't exist
        """
        if npc_id not in self.npcs:
            raise NPCNotFoundError(npc_id)

        npc = self.npcs[npc_id]
        return npc.chat(message, user_id)

    def get_npc_chat_history(self, npc_id: str, user_id: str) -> list:
        """Retrieve chat history between a specific user and NPC.

        Args:
            npc_id: The unique identifier of the NPC
            user_id: The unique identifier of the user

        Returns:
            List of message dictionaries containing the chat history

        Raises:
            NPCNotFoundError: If the specified NPC doesn't exist
        """
        if npc_id not in self.npcs:
            raise NPCNotFoundError(npc_id)

        return list(self.npcs[npc_id].chat_histories[user_id])

In [ ]:
from openai import OpenAI

# Nebius uses the same OpenAI() class, but with additional details
client = OpenAI(
    base_url="https://api.studio.nebius.ai/v1/",
    api_key=os.environ.get("NEBIUS_API_KEY"),
)

model = "meta-llama/Meta-Llama-3.1-405B-Instruct"

# Creating a factory
npc_factory = NPCFactory(client=client, model=model)

In [ ]:
# Register a user
user_id = npc_factory.register_user("Alice")

# Set user preffered language
preffered_language = "Old English"
npc_factory.set_user_language(user_id, preffered_language)

# Create an NPC
npc_id = npc_factory.register_npc(
    world_description="Medieval London, XIII century",
    character_description="A knight at Edward I's court",
    has_scratchpad=False
)

In [ ]:
def prettify_string(text, max_line_length=80):
    """Prints a string with line breaks at spaces to prevent horizontal scrolling.

    Args:
        text: The string to print.
        max_line_length: The maximum length of each line.
    """

    output_lines = []
    lines = text.split("\n")
    for line in lines:
        current_line = ""
        words = line.split()
        for word in words:
            if len(current_line) + len(word) + 1 <= max_line_length:
                current_line += word + " "
            else:
                output_lines.append(current_line.strip())
                current_line = word + " "
        output_lines.append(current_line.strip())  # Append the last line
    return "\n".join(output_lines)

In [ ]:
# We can hack our own code a bit and use translation features to test the system.

npc = npc_factory.npcs[npc_id]

message = "Hello, who are you and what brings you here?"
message_translated = # use localization feature of your NPC to translate the message
print(f"Translated message: {message_translated}")

response = npc_factory.chat_with_npc(
    npc_id,
    user_id,
    message_translated
)
print("Original answer")
print(prettify_string(response))
print("Translated answer")
print(prettify_string(npc.localize_input(response)))

##Задание 2. Перевод стихов
Магистр права, как известно, плохо переводит стихи. Получающиеся в результате стихи редко имеют хорошую рифму и ритм. Давайте попробуем наивно перевести какой-нибудь стишок про Шалтая-Болтая на выбранный вами язык:

In [ ]:
language = "Dutch"
poem = """Humpty Dumpty sat on a wall,
Humpty Dumpty had a great fall.
All the king's horses and all the king's men
Couldn't put Humpty together again."""

print(answer_with_llm(
    f"Translate the following children's rhyme to {language}\n{poem}"
))

Вряд ли это хороший перевод, потому что рифма совершенно нарушена.
Ваша задача — создать цепочку вызовов LLM для выполнения следующих шагов по переводу стихотворения:
1. Сделайте наивный дословный перевод, чтобы сохранить смысл
2. Перепишите перевод, чтобы сохранить рифму и ритм, но, возможно, потерять часть смысла.
3. Наконец, попросите редактора просмотреть оригинал и перевод и внести последние исправления.
Мы рекомендуем вам попытаться предложить своим магистрам права выполнять работу «Переводчик», «Редактор» и так далее.
Также выберите язык, который вы понимаете лучше всего, чтобы вы могли хорошо оценить результат (вы также можете изменить оригинал на другой язык)

In [ ]:
def translate_in_stages(input: str, language: str) -> str:
    pass

In [ ]:
translate_in_stages(poem, language)

##Задание 3. Поиск героя
В нашем королевстве существует очень формальный процесс утверждения героя для конкретного квеста.
Ваша задача — реализовать процесс утверждения с помощью вызовов LLM:
Вы получаете запрос на героя и описание героя, которого можно нанять для этого квеста.
**Шаг 1**. Убедитесь, что запрос формально корректен. Вы можете придумать свои идеи, но мы предлагаем следующие критерии:
- Там указано имя человека, запрашивающего героя, и дата;
- Есть описание того, кто будет снабжать героя деньгами и другими ресурсами;
- Есть причина, по которой нужен герой, какой-то квест или испытание;
- Имеет рекомендуемую квалификацию героя;
**Шаг 2**. Убедитесь, что проблема, которую пытается решить запрос, достаточна для того, чтобы действительно найти героя, или, возможно, автор может сделать это самостоятельно или найти более простое решение.
**Шаг 3**. Убедитесь, что описание героя соответствует квесту и требованиям, предъявляемым к герою.
Эти действия следует выполнять последовательно, один за другим. Если какой-то из этапов не удался, немедленно возвращайте отказ с обоснованием — вы не хотите тратить лишние вычислительные ресурсы на недостойные запросы! Если все три шага выполнены успешно, верните «принято».

In [ ]:
def check_hero_request(request_for_a_hero: str, hero_descripition: str) -> str:
    pass

In [ ]:
epic_request = """
Epic Hero Request
Name of Requestor: Lord Aeldric of the Silver Vale
Date: The 15th Day of Bloomrise, Year 1025 of the Dawnstar Calendar

Resource Provision:
The hero shall be provisioned by the Guild of Eternal Flame, a conclave of wealthy artificers and arcane financiers, who have pledged a sum of 500,000 gold crowns, enchanted arms and armor, rare tomes of forgotten magic, a sky-bound griffon steed, and a personal aide skilled in healing and reconnaissance. All resources shall be delivered at the Hall of Summoning in Ironhold.

Purpose of Request / Quest Description:
Darkness stirs in the Hollow Spine Mountains, where the Obsidian Serpent — an ancient beast thought long dead — has risen anew. Villages lie in ruin, and the skies turn black with ash. The hero is summoned to descend into the Abyssal Breach, recover the lost Emberheart Crystal, and seal the rift before the World Spine fractures and all realms fall into chaos.

Recommended Hero Qualifications:

Proven mastery in combat, both arcane and martial

Experience in surviving extreme environments and demonic incursions

Wisdom enough to resist corruption, and strength to slay without hesitation

Familiarity with ancient dialects and lost technologies

A heart unwavering in the face of despair, and a spirit unbreakable by shadow

Let the stars guide the right soul to answer. The fate of the realms balances on a blade’s edge.
"""

not_epic_request = """
Epic Hero Request
Name of Requestor: Steve
Date: 3rd of January 2025

Resource Provision:
I'll pay from my pocket

Quest Description:
I need someone to run to a supermarket for me, i'm hungry

Required qualification:
- Be very fast
- Be smart to buy good snacks.
"""

wrong_request = """
Hello, I need a mighty warrior to slay evil, thank you!
"""

In [ ]:
epic_hero = """
Hero Profile: Kaelen Thorne, the Ash-Wrought Sentinel

Forged in the fires of the Blistering Wars and tempered by years wandering the haunted ruins of the Old Kingdoms, Kaelen Thorne is a battle-scarred veteran clad in rune-etched obsidian armor. With one eye gifted by the Seers of Valemire—able to glimpse the truth behind illusions—and a blade forged from a fallen star, Kaelen walks the line between light and shadow.

Equal parts scholar and warrior, Kaelen speaks the tongues of forgotten realms and wields spells that twist the very air. Haunted but unyielding, Kaelen has turned away crowns and glory before—but for a quest that may decide the fate of all creation, the Sentinel rises once more.
"""

not_so_epic_hero = """
Tom the cat

Is a cat, supposed to catch mice, but can't really do it.
Has a lot of different surprising weapons and contraptions, but they always work against them.

Works for cat food.
"""

In [ ]:
check_hero_request(request_for_a_hero=epic_request, hero_descripition=epic_hero)

In [ ]:
check_hero_request(request_for_a_hero=epic_request, hero_descripition=not_so_epic_hero)

In [ ]:
check_hero_request(request_for_a_hero=not_epic_request, hero_descripition=epic_hero)

In [ ]:
check_hero_request(request_for_a_hero=wrong_request, hero_descripition=epic_hero)